In [ ]:
# Notebook imports
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

In [ ]:
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Notebook configuration
plots_dir = PLOTS_DIR
plots_dir.mkdir(parents=True, exist_ok=True)

COLORS = {
    "price": "#2563eb",
    "volume": "#7c3aed",
    "anomaly": "#dc2626",
    "spread": "#059669",
    "normal": "#6b7280",
    "highlight": "#f59e0b",
}

In [ ]:
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Self-load: ensure data is available
from pathlib import Path

PROCESSED_CSV = DATA_DIR / "processed" / "msft_hourly(in)_processed.csv"
if "df" not in globals() or "quote_datetime" not in df.columns:
    if not PROCESSED_CSV.exists():
        raise FileNotFoundError(
            f"Processed CSV not found at {PROCESSED_CSV}. "
            "Run 1.0-abz-preprocessing.ipynb first to generate it."
        )
    df = pd.read_csv(PROCESSED_CSV, parse_dates=["quote_datetime"])

if "plots_dir" not in globals():
    plots_dir = PLOTS_DIR
    plots_dir.mkdir(parents=True, exist_ok=True)

if "COLORS" not in globals():
    COLORS = {
        "price": "#2563eb",
        "volume": "#7c3aed",
        "anomaly": "#dc2626",
        "spread": "#059669",
        "normal": "#6b7280",
        "highlight": "#f59e0b",
    }

if "quote_datetime" not in df.columns:
    raise RuntimeError("df missing 'quote_datetime'. Check data file.")
if not pd.api.types.is_datetime64_any_dtype(df["quote_datetime"]):
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])
print("Prerequisites OK. Proceed with EDA.")

### 4a. Full Time Series Plot

In [ ]:
# Create subplots
fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "Close Price",
        "Open Price",
        "High Price",
        "Low Price",
        "Mid Price",
    ),
    row_heights=[0.16, 0.16, 0.16, 0.16, 0.16],  # Increased the volume row height
)

price_configs = [
    ("close", COLORS["price"]),
    ("open", COLORS["spread"]),
    ("high", COLORS["anomaly"]),
    ("low", COLORS["normal"]),
    ("mid", COLORS["highlight"]),
]

for i, (col, color) in enumerate(price_configs, 1):
    # Add the main price line
    fig.add_trace(
        go.Scatter(
            x=df["quote_datetime"],
            y=df[col],
            name=col.title(),
            line=dict(color=color, width=1),
            opacity=0.7,
        ),
        row=i,
        col=1,
    )
    # Add range shading for the close price plot
    if col == "close":
        fig.add_trace(
            go.Scatter(
                x=pd.concat([df["quote_datetime"], df["quote_datetime"][::-1]]),
                y=pd.concat([df["high"], df["low"][::-1]]),
                fill="toself",
                fillcolor="rgba(37, 99, 235, 0.1)",
                line=dict(color="rgba(255,255,255,0)"),
                name="High-Low Range",
                opacity=0.7,
                showlegend=True,
            ),
            row=1,
            col=1,
        )


fig.update_layout(
    height=1400,  # Increased total height for better visibility of all subplots
    template="plotly_white",
    title_text="MSFT Interactive Market Analysis (Hourly)",
    showlegend=True,
    hovermode="x unified",
)

save_path = plots_dir / "fig1_msft_price_analysis.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

## Chart Interpretation: Price
The above charts visually confirm massive multi-year uptrend. High-volatility eras (like early 2020) are visible.

### 4b. Price Distribution

In [ ]:
# OHLC Distributions
fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=([f"{c.upper()} Price" for c in ["open", "high", "low", "close"]]),
)

for i, col in enumerate(["open", "high", "low", "close"], 1):
    fig.add_trace(
        go.Histogram(x=df[col], nbinsx=60, name=col, marker_color=COLORS["price"]),
        row=1,
        col=i,
    )

fig.update_layout(
    height=400,
    title_text="OHLC Price Distributions",
    template="plotly_white",
    xaxis_title="Price",
    yaxis_title="Frequency",
    showlegend=False,
)

save_path = plots_dir / "fig2_ohlc_price_distributions.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

In [ ]:
metrics = ["open", "high", "low", "close", "spread_pct", "vwap_deviation", "return_1h"]
existing_metrics = [m for m in metrics if m in df.columns]

# Create a multi-pane box plot
fig = make_subplots(
    rows=1,
    cols=len(existing_metrics),
    subplot_titles=[m.replace("_", " ").title() for m in existing_metrics],
)

for i, col in enumerate(existing_metrics, 1):
    fig.add_trace(
        go.Box(y=df[col], name=col, marker_color=COLORS["price"], boxpoints="outliers"),
        row=1,
        col=i,
    )

fig.update_layout(
    height=500,
    title_text="Statistical Outlier Analysis: Metric Box Plots",
    template="plotly_white",
    showlegend=False,
)

save_path = plots_dir / "fig3_ohlc_metric_box_plots.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

## Chart Interpretation: Price Distributions
The histograms and box plots, together, provide a comprehensive view of the OHLC price distributions.

Histograms visually confirm that the MSFT's 9-year price history is heavily **right-skewed**, rather than normally distributed. This means the majority of trading activity occurred at lower historical price points, with fewer occurrences at very high prices. The mean (dashed red line) consistently appears to the right of the peak of the distribution, pulled by these higher values, and to the right of the median (dashed yellow line).

Box plots further illustrate this skewness. While they show the median, interquartile range, and potential outliers, for right-skewed data, we would typically observe the median closer to the bottom of the box (25th percentile), and a longer upper whisker compared to the lower whisker, indicating the spread of higher values. Both plots collectively highlight that the price data does not follow a symmetrical normal distribution.

This finding is crucial for anomaly detection. It demonstrates that naive methods relying on assumptions of normal distribution, such as fixed standard deviation thresholds, would be unsuitable. Instead, it validates the use of advanced spatial algorithms like `IsolationForest` that can effectively identify anomalies in non-normally distributed, skewed data by focusing on how isolated data points are from dense clusters of 'normal' behavior, irrespective of their absolute magnitude.

### 4c. Year-over-Year Price Comparison

In [ ]:
df["year"] = df["quote_datetime"].dt.year
df["day_of_year"] = df["quote_datetime"].dt.dayofyear

fig = go.Figure()

for year in sorted(df["year"].unique()):
    year_data = df[df["year"] == year]
    fig.add_trace(
        go.Scatter(
            x=year_data["day_of_year"],
            y=year_data["close"],
            name=str(year),
            mode="lines",
            line=dict(width=1.5),
            hovertemplate="Day: %{x}<br>Price: $%{y:.2f}",
        )
    )

fig.update_layout(
    title="Year-over-Year Price Comparison",
    xaxis_title="Day of Year",
    yaxis_title="Close Price ($)",
    template="plotly_white",
    hovermode="closest",
    height=600,
)

save_path = plots_dir / "fig4_yoy_price_comparison.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

The **'Year-over-Year Price Comparison'** plot shows how MSFT's closing price has performed throughout the year, with each line representing a different year.

Key observations from this plot include:

**Long-term Uptrend:** The plot clearly illustrates the overall upward trend in MSFT's stock price over the years included in the dataset (2017-2026). Each subsequent year's line generally starts higher or reaches higher peaks than the previous year, indicating consistent growth.

**Seasonal Patterns:** By aligning prices by 'Day of Year', we can observe if there are consistent seasonal trends in price movements. For example, some years might show similar patterns of dips or rallies around certain times of the year, though the magnitude can vary greatly.

**Relative Performance:** We can quickly compare the performance of different years. For instance, we might see how 2025's price trajectory initially dipped but then recovered strongly. The plot helps identify periods of significant outperformance or underperformance relative to other years.

**Volatility Changes:** The spread between the lines at different points in the year can also give a visual sense of varying volatility across years or within a year.

---
## 5. Volume Analysis

In [ ]:
# Volume Analysis
fig = make_subplots(
    rows=1, cols=3, subplot_titles=("Trade Volume", "Log Volume", "Avg Volume by Hour")
)

fig.add_trace(
    go.Histogram(
        x=df["trade_volume"], nbinsx=80, marker_color=COLORS["volume"], opacity=0.7
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=np.log1p(df["trade_volume"]),
        nbinsx=80,
        marker_color=COLORS["volume"],
        opacity=0.7,
    ),
    row=1,
    col=2,
)

hourly_vol = df.groupby(df["quote_datetime"].dt.hour)["trade_volume"].mean()
fig.add_trace(
    go.Bar(
        x=hourly_vol.index,
        y=hourly_vol.values,
        marker_color=COLORS["volume"],
        opacity=0.7,
    ),
    row=1,
    col=3,
)

fig.update_layout(
    height=400,
    title_text="Volume Seasonality Analysis",
    template="plotly_white",
    showlegend=False,
)

save_path = plots_dir / "fig5_volume_analysis.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

## Chart Interpretation: Volume Seasonality
The **left chart** gives an absolute distribution of trade volume. The **center chart** normalizes raw trade volume using a logarithmic scale `np.log1p()`, revealing a clean bell-curve distribution of market participation. The **right chart** highlights the intraday 'Volatility Smile (Implied volatility patterns that arise in pricing financial options)' : volume predictably spikes at the market Open (9:30 AM) and Close (4:00 PM). Our ML models must be aware that high volume at these specific hours is *expected behavior*, not an anomaly.

---
## 6. Bid-Ask Spread Analysis
The **Bid-Ask Spread** is a key measure of market liquidity. A wider spread indicates lower liquidity (fewer buyers and sellers, or higher risk perceived by market makers), while a narrower spread indicates higher liquidity. During times of market distress (anomalies), market makers pull their quotes, and the spread dramatically widens. This makes it an excellent feature for identifying 'flash crashes' or severe panic. We calculate this as a **Percentage (`spread_pct`)** rather than a raw dollar amount as a 0.05 spread when MSFT is at 40 is massive (0.125%), but when it is at 200, a 0.05 spread is tiny (0.025%). Standardizing this as a percentage ensures the ML model treats the risk equally across the entire 9-year timeline.

In [ ]:
# Bid-Ask Spread Analysis
if "bid" in df.columns and "ask" in df.columns:
    df["spread"] = df["ask"] - df["bid"]
    df["spread_pct"] = df["spread"] / ((df["ask"] + df["bid"]) / 2) * 100

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            "Spread % Over Time",
            "Spread % Distribution",
            "Avg Spread % by Hour",
        ),
    )

    # 1) Time series
    fig.add_trace(
        go.Scatter(
            x=df["quote_datetime"],
            y=df["spread_pct"],
            mode="lines",
            line=dict(color=COLORS["spread"], width=0.8),
            opacity=0.7,
            name="Spread %",
        ),
        row=1,
        col=1,
    )

    # 2) Histogram
    fig.add_trace(
        go.Histogram(
            x=df["spread_pct"],
            nbinsx=60,
            marker_color=COLORS["spread"],
            opacity=0.7,
            name="Spread % Distribution",
        ),
        row=1,
        col=2,
    )

    # 3) Hourly average
    hourly_spread = df.groupby(df["quote_datetime"].dt.hour)["spread_pct"].mean()
    fig.add_trace(
        go.Bar(
            x=hourly_spread.index,
            y=hourly_spread.values,
            marker_color=COLORS["spread"],
            opacity=0.7,
            name="Avg Spread %",
        ),
        row=1,
        col=3,
    )

    fig.update_layout(
        height=450,
        title_text="Bid-Ask Spread Analysis",
        template="plotly_white",
        showlegend=False,
    )
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Spread %", row=1, col=2)
    fig.update_xaxes(title_text="Hour of Day", row=1, col=3)
    fig.update_yaxes(title_text="Spread %", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=1, col=2)
    fig.update_yaxes(title_text="Avg Spread %", row=1, col=3)

    save_path = plots_dir / "fig6_bid-ask_spread_analysis.png"
    if not save_path.exists():
        fig.write_image(save_path, width=1600, height=900, scale=2)
        print(f"Saved: {save_path}")
    else:
        print(f"File already exists, skipped: {save_path}")

    fig.show()

## Chart Interpretation: Liquidity Vacuums
The **time-series plot (left)** reveals extreme liquidity vacuums (massive spikes in the spread) during systemic market crashes. The **histogram (middle)** proves that 99% of the time, the spread is incredibly tight (near 0%). The **seasonality bar chart (right)** proves spreads are widest during illiquid overnight or pre-market hours. A wider spread at market open or close might be considered normal due to higher volatility and order imbalances. However, a sudden, massive spike in the bid-ask spread during typical mid-day hours (like around 12:00 PM), when liquidity (and trading activity) is usually high and spreads are tight, would be a strong signal of a potential market anomaly or 'liquidity vacuum (area with minimal trading volume)'. It suggests unusual market stress or a sudden lack of participants willing to trade at normal prices.

In essence, this chart helps to establish a baseline of **'normal'** hourly liquidity patterns, allowing us to identify when the bid-ask spread deviates significantly from its expected range for a given time of day, thereby flagging potential anomalies.

---
## 7. Volume-Weighted Average Price (VWAP) Analysis
It represents the average price a stock has traded at throughout the day, weighted by volume. It's often used by institutional traders to evaluate their execution quality; they aim to buy below VWAP and sell above it.

In [ ]:
if "vwap" in df.columns:
    df["vwap_deviation"] = (df["close"] - df["vwap"]) / df["vwap"] * 100

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Close vs VWAP (First 30 Days)",
            "VWAP Deviation % Distribution",
        ),
    )

    # VWAP vs Close (First 30 days roughly 210 bars)
    sample = df.head(210)
    fig.add_trace(
        go.Scatter(
            x=sample["quote_datetime"],
            y=sample["close"],
            name="Close",
            line=dict(color=COLORS["price"]),
            opacity=0.7,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=sample["quote_datetime"],
            y=sample["vwap"],
            name="VWAP",
            line=dict(color=COLORS["highlight"], dash="dash"),
            opacity=0.7,
        ),
        row=1,
        col=1,
    )

    # VWAP deviation distribution
    fig.add_trace(
        go.Histogram(
            x=df["vwap_deviation"],
            nbinsx=80,
            name="Deviation %",
            marker_color=COLORS["highlight"],
            opacity=0.7,
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        height=450, title_text="VWAP Analysis Dashboard", template="plotly_white"
    )

    save_path = plots_dir / "fig7_vwap_analysis.png"
    if not save_path.exists():
        fig.write_image(save_path, width=1600, height=900, scale=2)
        print(f"Saved: {save_path}")
    else:
        print(f"File already exists, skipped: {save_path}")

    fig.show()

The 'VWAP Analysis' section provides insights into how the stock's closing price deviates from its Volume-Weighted Average Price (VWAP). This deviation is a key indicator for understanding trading imbalances and potential anomalies.

1. **Close vs VWAP:** This plot visually compares the hourly closing price to the VWAP for first 30 days. Ideally, the close price should hover closely around the VWAP. Significant deviations can indicate strong buying or selling pressure. For these 30 days, we can observe how closely the close price tracks the VWAP. Short-term movements away from VWAP followed by a return towards it often indicate mean-reverting behavior, while prolonged deviations might suggest sustained market trends or unusual activity.

2. **VWAP Deviation % Distribution:** This histogram shows the distribution of the percentage deviation of the Close price from the VWAP ((Close - VWAP) / VWAP * 100). By normalizing the deviation as a percentage, we can compare volatility and price action across different price levels over the entire 8-year dataset. It helps to understand the typical range of deviation and identify extreme occurrences. The histogram reveals how often the close price is near the VWAP (indicated by the peak around 0%).

**Severe Deviations' and 'Unsustainable Exhaustion Points**:

1. The **'long tails'** of the histogram represent instances where the close price significantly deviates from the VWAP. These are less frequent but more extreme occurrences. Such large deviations are often considered 'unsustainable exhaustion points' because, in a mean-reverting market, prices are expected to eventually correct back towards the VWAP. A price that moves extremely far from VWAP suggests either very strong, sustained buying/selling pressure or an unusual market event.

    *  **Algorithmic Mean-Reversion:** This oscillating behavior is often attributed to large institutional trading algorithms. These algorithms frequently use VWAP as a benchmark, aiming to execute large orders close to or better than the prevailing VWAP. If the price moves too far above VWAP, these algorithms might see it as an opportunity to sell; if it moves too far below, they might see it as an opportunity to buy. This constant 'gravitational pull' towards VWAP is known as mean-reversion.

2. The key takeaway for anomaly detection is that these severe deviations from the VWAP are "excellent non-linear features" for the anomaly engine. While small fluctuations around VWAP are normal and expected (mean-reversion), unusually large or prolonged deviations signal abnormal market conditions. These could be due to:

    *   Flash events: Sudden, rapid price movements.
    *   Order imbalances: One side (buy or sell) overwhelmingly dominates.
    *   Information asymmetry: Some market participants having information not yet reflected in the VWAP.

In [ ]:
if "vwap_deviation" in df.columns:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df["quote_datetime"],
            y=df["vwap_deviation"],
            name="VWAP Deviation",
            line=dict(color=COLORS["highlight"], width=1),
            opacity=0.7,
        )
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.update_layout(
        title="Close Price % Deviation from VWAP Over Time",
        xaxis_title="Date",
        yaxis_title="Deviation (%)",
        template="plotly_white",
        height=500,
    )

    save_path = plots_dir / "fig8_close-vwap_deviation.png"
    if not save_path.exists():
        fig.write_image(save_path, width=1600, height=900, scale=2)
        print(f"Saved: {save_path}")
    else:
        print(f"File already exists, skipped: {save_path}")

    fig.show()
else:
    print("Column 'vwap_deviation' not found.")

**Anomalous Spikes:** The extreme vertical spikes (some reaching several percentage points away from 0) represent **exhaustion points**. These are moments of intense buying or selling pressure that were unsustainable, making them perfect mathematical features for our Isolation Forest and LSTM models to flag as anomalies.

---
## 8. Correlation Analysis

In [ ]:
# Prepare Data
corr_cols = [
    "open",
    "high",
    "low",
    "close",
    "trade_volume",
    "vwap",
    "bid",
    "ask",
    "mid",
    "spread_pct",
    "vwap_deviation",
    "return_1h",
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr().round(2)

# Heatmap
fig = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale="RdBu_r",
    showscale=True,
)

fig.update_layout(
    title_text="Correlation Matrix (Price & Quote Variables)", template="plotly_white"
)

save_path = plots_dir / "fig9_correlation_matrix.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

## Chart Interpretation: Feature Orthogonality
This heatmap reveals deep multi-collinearity between raw price features (Open, High, Low, Close are ~1.0 perfectly correlated). Feeding all four into a model is redundant and introduces bias. However, our engineered features (`spread_pct`, `vwap_deviation`, `return_1h`), exhibit near-zero correlation with raw price, proving we have successfully extracted independent, highly orthogonal signals for the Machine Learning model to evaluate.

---
## 9. Seasonality & Periodicity Patterns

In [ ]:
# Calculate 1h returns first to avoid KeyError
df["return_1h"] = df["close"].pct_change()

# Seasonality & Time Patterns

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Avg 1h Return by Hour",
        "Return Volatility by Hour",
        "Avg 1h Return by Day",
        "Avg 1h Return by Month",
    ),
)

# Hourly Return
hourly_ret = df.groupby(df["quote_datetime"].dt.hour)["return_1h"].mean()
fig.add_trace(
    go.Bar(
        x=hourly_ret.index,
        y=hourly_ret.values,
        marker_color=COLORS["price"],
        opacity=0.7,
    ),
    row=1,
    col=1,
)

# Hourly Volatility
hourly_std = df.groupby(df["quote_datetime"].dt.hour)["return_1h"].std()
fig.add_trace(
    go.Bar(
        x=hourly_std.index,
        y=hourly_std.values,
        marker_color=COLORS["anomaly"],
        opacity=0.7,
    ),
    row=1,
    col=2,
)

# Day of Week
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily_ret = df.groupby(df["quote_datetime"].dt.dayofweek)["return_1h"].mean()
fig.add_trace(
    go.Bar(x=day_names, y=daily_ret.values, marker_color=COLORS["price"], opacity=0.7),
    row=2,
    col=1,
)

# Monthly
month_names = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
monthly_ret = df.groupby(df["quote_datetime"].dt.month)["return_1h"].mean()
fig.add_trace(
    go.Bar(
        x=month_names, y=monthly_ret.values, marker_color=COLORS["price"], opacity=0.7
    ),
    row=2,
    col=2,
)

fig.update_layout(
    height=800,
    title_text="Seasonality Analysis",
    template="plotly_white",
    showlegend=False,
)

save_path = plots_dir / "fig10_seasonality_and_periodicity_patterns.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

In [ ]:
# Two stacked plots for 11:00 hour only:
# 1) Percentage hourly return: close.pct_change()
# 2) Signed hourly return in price units: close.diff()

if "return_1h" not in df.columns:
    df["return_1h"] = df["close"].pct_change()

if "return_1h_signed" not in df.columns:
    df["return_1h_signed"] = df["close"].diff()

hr11 = df[df["quote_datetime"].dt.hour == 11].copy()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "11:00 Hourly Return (%) - close.pct_change()",
        "11:00 Hourly Signed Return (Price Units) - close.diff()",
    ),
)

# Plot 1: 11:00 percent return
fig.add_trace(
    go.Scatter(
        x=hr11["quote_datetime"],
        y=hr11["return_1h"],
        mode="lines+markers",
        name="11:00 return_1h (%)",
        line=dict(color=COLORS["price"], width=1.2),
        marker=dict(size=4, opacity=0.8),
        hovertemplate="Date: %{x}<br>Return: %{y:.4%}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_hline(y=0, line_dash="dash", line_color=COLORS["anomaly"], row=1, col=1)

# Plot 2: 11:00 signed price return (not percent)
fig.add_trace(
    go.Scatter(
        x=hr11["quote_datetime"],
        y=hr11["return_1h_signed"],
        mode="lines+markers",
        name="11:00 signed return (price)",
        line=dict(color=COLORS["highlight"], width=1.2),
        marker=dict(size=4, opacity=0.8),
        hovertemplate="Date: %{x}<br>Signed Return: %{y:.4f}<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_hline(y=0, line_dash="dash", line_color=COLORS["anomaly"], row=2, col=1)

fig.update_yaxes(title_text="Return (%)", tickformat=".2%", row=1, col=1)
fig.update_yaxes(title_text="Signed Return (Price)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.update_layout(
    title="MSFT 11:00 Hour Returns: Percent vs Signed Price Change",
    template="plotly_white",
    height=800,
    hovermode="x unified",
)

fig.show()

In [ ]:
# Average signed hourly return for 11:00 (price units)
if "return_1h_signed" not in df.columns:
    df["return_1h_signed"] = df["close"].diff()

hr11 = df[df["quote_datetime"].dt.hour == 11].copy()
avg_hourly_signed_return_11 = hr11["return_1h_signed"].mean()

print(
    "Average signed hourly return at 11:00 (price units):", avg_hourly_signed_return_11
)

In [ ]:
avg_hourly_signed_return_11 = (
    df.loc[df["quote_datetime"].dt.hour == 11, "close"].diff().mean()
)
avg_hourly_signed_return_11

The **'Seasonality & Periodicity Patterns'** section reveals several key findings from the plots regarding MSFT's hourly returns across different timeframes:

1. **Average 1h Return by Hour of Day (Top-Left Plot)**: This plot identifies if certain hours consistently yield higher or lower returns on average, with error bars showing return variability.

2. **Return Volatility (Std) by Hour of Day (Top-Right Plot):** This plot clearly demonstrates the 'volatility smile', where volatility is significantly higher at market open (around 10:00 AM) and market close (15:00-16:00) compared to mid-day hours. This is crucial because a large price move during a low-volatility hour (e.g., 12:00 PM) is considered more anomalous than the same move during a high-volatility hour.

3. **Average 1h Return by Day of Week (Bottom-Left Plot):** This chart highlights potential weekly return patterns, showing whether returns tend to be different on certain days like Mondays or Fridays.

4. **Average 1h Return by Month (Bottom-Right Plot):** This plot uncovers monthly seasonal trends, indicating periods of typically stronger (e.g., June) or weaker (e.g., September) performance, potentially influenced by recurring events.

**Overall Key Findings: The Volatility Smile and its Impact on Anomaly Detection**

The Volatility Smile', is the classic 'U-Shape' of equities: Opening and Closing hours are immensely more volatile than the midday doldrums. This seasonality is paramount for anomaly detection. It establishes a baseline of 'normal' behavior, meaning a volatility spike or extreme price movement at 12:00 PM is significantly more anomalous and signals distress more strongly than an identical spike occurring at 9:30 AM (market open). Understanding these predictable market rhythms allows the model to differentiate between expected fluctuations and truly anomalous events.

---
## 10. Time Gap Analysis

 This section provides insights into the time intervals between consecutive data points, which is crucial for understanding the regularity and completeness of the hourly data.

In [ ]:
time_diffs = df["quote_datetime"].diff().dropna()
time_diffs_hours = time_diffs.dt.total_seconds() / 3600

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Distribution of Time Gaps (< 24h)",
        "Unusual Time Gaps (1.5h-14h)",
    ),
)

# Main gaps
fig.add_trace(
    go.Histogram(
        x=time_diffs_hours[time_diffs_hours < 24],
        nbinsx=50,
        name="Gaps",
        marker_color=COLORS["price"],
        opacity=0.7,
    ),
    row=1,
    col=1,
)
fig.add_vline(
    x=1.0,
    line_dash="dash",
    line_color=COLORS["anomaly"],
    annotation_text="Expected: 1h",
    row=1,
    col=1,
)

# Unusual gaps
unusual = time_diffs_hours[(time_diffs_hours > 1.5) & (time_diffs_hours < 14)]
if len(unusual) > 0:
    fig.add_trace(
        go.Histogram(
            x=unusual,
            nbinsx=30,
            name="Unusual",
            marker_color=COLORS["anomaly"],
            opacity=0.7,
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    height=450,
    title_text="Time Gap Analysis",
    template="plotly_white",
    showlegend=False,
)

save_path = plots_dir / "fig11_time_gap_analysis.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

## Chart Interpretation:

1. **Distribution of Time Gaps (< 24h) (Left Plot):**
This is a histogram that visualizes the frequency of different time durations between consecutive data bars, specifically focusing on gaps shorter than 24 hours. An expected vertical line at 1.0h is often present, representing the typical hourly frequency of the data. For hourly stock data, we expect to see a dominant peak at 1.0 hour, indicating regular hourly observations. The presence of other peaks, especially around 16 hours, typically represents the overnight closing period when the market is not trading. This plot confirms the primary frequency of the data and identifies common market closures within a day.

2. **Unusual Time Gaps (Right Plot):**
This histogram specifically focuses on time gaps that are neither the standard 1-hour interval nor the typical overnight or weekend gaps. It's looking for durations between, for example, 1.5 hours and 14 hours, which would be considered 'unusual' for a consistently collected hourly dataset.

---
## 11. Rolling Volatility Regime Analysis

In [ ]:
# Rolling Volatility Regime Analysis
df["vol_24h"] = df["return_1h"].rolling(24).std()
df["vol_120h"] = df["return_1h"].rolling(120).std()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("MSFT Price", "Rolling Volatility Regimes"),
)

# Price Trace
fig.add_trace(
    go.Scatter(
        x=df["quote_datetime"],
        y=df["close"],
        name="Close",
        line=dict(color=COLORS["price"], width=1),
        opacity=0.7,
    ),
    row=1,
    col=1,
)

# Volatility Traces
fig.add_trace(
    go.Scatter(
        x=df["quote_datetime"],
        y=df["vol_24h"],
        name="24h Vol",
        line=dict(color=COLORS["anomaly"], width=1),
        opacity=0.7,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=df["quote_datetime"],
        y=df["vol_120h"],
        name="1-Week Vol",
        line=dict(color=COLORS["highlight"], width=1.5),
        opacity=0.7,
    ),
    row=2,
    col=1,
)

fig.update_layout(
    height=700,
    title_text="Volatility Regime Analysis Dashboard",
    template="plotly_white",
)

save_path = plots_dir / "fig12_volatility_regime_analysis.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

### Key Findings from the Plots:

1. **Volatility Clustering:** The plots clearly illustrate the phenomenon of volatility clustering. This means that periods of high market turbulence (high volatility) tend to be followed by more high volatility, and similarly, calm periods tend to persist.
2. **Multiscale Volatility:** The analysis uses different rolling windows (24-hour and 1-week) to capture both short-term and more sustained volatility. Spikes in these lines correspond to periods of increased price fluctuations.
3. **Identification of High-Volatility Regimes:** when 1-week rolling volatility is significantly elevated. These regions pinpoint historical periods of significant market stress, such as the COVID-19 crash, where price movements are naturally larger.

**Critical context for anomaly detection:** what's "anomalous" during a calm period might be
"normal" during a crisis.

In [ ]:
# Calculate 24h rolling std for the difference plot
rolling_std = df["return_1h"].rolling(window=24).std()
diff_returns_std = df["return_1h"] - rolling_std

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("MSFT Hourly Returns", "Difference: Return_1h - Rolling Std (24h)"),
)

fig.add_trace(
    go.Scatter(
        x=df["quote_datetime"],
        y=df["return_1h"],
        name="1h Return",
        line=dict(color=COLORS["price"], width=0.8),
        opacity=0.7,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=df["quote_datetime"],
        y=diff_returns_std,
        name="Return - Vol",
        line=dict(color=COLORS["anomaly"], width=0.8),
        opacity=0.7,
    ),
    row=2,
    col=1,
)

fig.update_layout(
    height=800,
    title_text="Volatility Analysis Dashboard",
    template="plotly_white",
    showlegend=True,
)

save_path = plots_dir / "fig13_volatility_analysis_dashboard.png"
if not save_path.exists():
    fig.write_image(save_path, width=1600, height=900, scale=2)
    print(f"Saved: {save_path}")
else:
    print(f"File already exists, skipped: {save_path}")

fig.show()

---
## 12. COVID Crash Zoom-In (Feb - Apr 2020)

The 'COVID Crash Zoom-In' section focuses on the period between February and April 2020 to analyze the impact of the COVID-19 pandemic on AAPL stock, which was the most significant market event in the dataset.

In [ ]:
covid_df = df[
    (df["quote_datetime"] >= "2020-02-15") & (df["quote_datetime"] <= "2020-04-20")
]

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "COVID-19 Market Crash: Price Action",
        "Hourly Returns",
        "Bid-Ask Spread",
    ),
)

# Price + Range
fig.add_trace(
    go.Scatter(
        x=covid_df["quote_datetime"],
        y=covid_df["close"],
        name="Close",
        line=dict(color=COLORS["price"]),
        opacity=0.7,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=pd.concat([covid_df["quote_datetime"], covid_df["quote_datetime"][::-1]]),
        y=pd.concat([covid_df["high"], covid_df["low"][::-1]]),
        fill="toself",
        fillcolor="rgba(37, 99, 235, 0.1)",
        line=dict(color="rgba(255,255,255,0)"),
        name="High-Low Range",
        opacity=0.7,
    ),
    row=1,
    col=1,
)

# Returns (Colored by sign)
fig.add_trace(
    go.Bar(
        x=covid_df["quote_datetime"],
        y=covid_df["return_1h"],
        name="Return",
        marker_color=[
            COLORS["anomaly"] if r < 0 else COLORS["spread"]
            for r in covid_df["return_1h"]
        ],
        opacity=0.7,
    ),
    row=2,
    col=1,
)

# Spread
fig.add_trace(
    go.Scatter(
        x=covid_df["quote_datetime"],
        y=covid_df["spread_pct"],
        name="Spread %",
        line=dict(color=COLORS["highlight"]),
        opacity=0.7,
    ),
    row=3,
    col=1,
)

fig.update_layout(
    height=900, title_text="Detailed COVID-19 Impact Analysis", template="plotly_white"
)
fig.show()

### Key Findings from the Plots:

1. **Price Action (Top Plot):** This plot clearly shows the dramatic and rapid decline in MSFT's price during this period. The annotations highlight the pre-crash peak and the subsequent trough, illustrating a significant **~28.5%** drop from **February 19, 2020**, to **March 23, 2020**. The shaded area representing the high-low range also indicates increased volatility during this downturn.
2. **Hourly Returns (Middle Plot):** The bar chart of hourly returns during the COVID crash vividly displays the heightened volatility and frequent large price swings. There's a clear prevalence of large negative returns (red bars), indicative of intense selling pressure, interspersed with some significant positive returns (green bars) as the market attempted to rebound or stabilize.
3. **Bid-Ask Spread (Bottom Plot):** This plot shows a noticeable widening of the bid-ask spread during the crash. A wider spread signifies a significant reduction in market liquidity, which is a classic symptom of market stress and uncertainty. This occurs as market makers pull back, and fewer participants are willing to take on risk, leading to larger gaps between buying and selling prices.

**Overall Significance for Anomaly Detection:** This section serves as a crucial validation point for any anomaly detection model. The model must be able to identify this period of extreme market distress as anomalous to be considered credible. The combination of sharp price declines, highly volatile returns, and widened bid-ask spreads provides strong, multi-faceted signals of an anomaly that the pipeline is designed to detect.